# Modelado

En esta etapa se construirá el modelo de clasificación de reseñas utilizando técnicas de Procesamiento de Lenguaje Natural (NLP) y Deep Learning. A partir del conjunto de datos previamente preprocesado, las reseñas serán transformadas mediante la representación numérica **TF-IDF**, permitiendo convertir el texto en vectores adecuados para el entrenamiento de un modelo neuronal.

Posteriormente, se implementará un clasificador utilizando **PyTorch**, compuesto por capas completamente conectadas (*fully connected*), Batch Normalization y Dropout para mejorar la estabilidad del entrenamiento y reducir el sobreajuste. El proceso incluirá la división del conjunto de datos, el entrenamiento del modelo, la evaluación de su desempeño y el análisis de las métricas obtenidas.

El objetivo final es desarrollar un modelo capaz de predecir la calificación (*Rating*) de una reseña a partir de su contenido textual, evaluando su capacidad de generalización sobre datos no vistos.

## Importación de librerías y carga del conjunto de datos

Se importan las bibliotecas necesarias para el modelado y se carga el conjunto de datos previamente preprocesado, el cual será utilizado para la representación numérica mediante TF-IDF y el entrenamiento del modelo de clasificación.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [13]:
import importlib
import src.utils.seed as seed

importlib.reload(seed)

<module 'src.utils.seed' from 'c:\\Users\\lauta\\Desktop\\DataScience\\amazon_reviews_nlp\\src\\utils\\seed.py'>

In [14]:
import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split

from src.utils.seed import set_seed

RANDOM_STATE = 42

set_seed(RANDOM_STATE)

In [5]:
DATA_PATH = Path("../data/processed/amazon_reviews_processed.csv")

df = pd.read_csv(DATA_PATH)

display(df.head())

,Text,Rating,Age,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,absolutely wonderful - silky sexy comfortable,4,33,1,0,Initmates,Intimate,Intimates
1,love dress ! sooo pretty . happened find store...,5,34,1,4,General,Dresses,Dresses
2,major design flaw high hope dress really wante...,3,60,0,0,General,Dresses,Dresses
3,"favorite buy ! love , love , love jumpsuit . f...",5,50,1,0,General Petite,Bottoms,Pants
4,flattering shirt shirt flattering due adjustab...,5,47,1,6,General,Tops,Blouses


In [15]:
print(f"Observaciones: {len(df):,}")
print(f"Variables: {df.shape[1]}")

Observaciones: 22,641
Variables: 8


## Representación numérica mediante TF-IDF

Los algoritmos de aprendizaje automático no pueden trabajar directamente con texto, por lo que resulta necesario transformar las reseñas en una representación numérica.

En este proyecto se emplea **TF-IDF (Term Frequency - Inverse Document Frequency)**, una técnica que pondera la importancia de cada palabra dentro de un documento considerando también su frecuencia en todo el corpus. De esta forma, los términos muy frecuentes reciben un menor peso, mientras que aquellos más representativos adquieren mayor relevancia.

Esta representación resulta más adecuada que un conteo simple de palabras (*Bag of Words*), ya que permite reducir la influencia de términos poco informativos y mejorar la capacidad discriminativa del modelo de clasificación.

In [18]:
from src.features.tfidf import build_tfidf

MAX_FEATURES = 5000

vectorizer = build_tfidf(MAX_FEATURES)

X = df["Text"]
y = df["Rating"]

X_tfidf = vectorizer.fit_transform(X)

print(f"Número de documentos: {X_tfidf.shape[0]:,}")
print(f"Número de características: {X_tfidf.shape[1]:,}")


Número de documentos: 22,641
Número de características: 5,000


In [22]:
display(
    pd.Series(feature_names)
    .sample(20, random_state=RANDOM_STATE)
    .to_frame(name="Términos del vocabulario")
)

,Términos del vocabulario
1501,feel good
2586,lunch
2653,material thin
1055,definitely run
705,class
106,6p
589,bunched
2468,looking
2413,longer front
1600,fit tight


### Conclusiones

Las reseñas fueron transformadas exitosamente a una representación numérica mediante TF-IDF, obteniendo una matriz de **22.641 documentos** y un vocabulario de **5.000 características**. Esta representación reduce la influencia de palabras muy frecuentes y conserva los términos más representativos del corpus, proporcionando una entrada adecuada para el entrenamiento del modelo de clasificación.

Se utilizará un vocabulario máximo de 5.000 términos, incluyendo unigramas y bigramas. Además, se eliminarán palabras extremadamente raras o excesivamente frecuentes para mejorar la calidad de la representación numérica.